# 01 — Dataset Preparation Colab

This notebook takes your `prepared_dataset.zip`, finds the combined dataset table, validates the current multilingual categories, maps categories to the 12 target labels, removes normalized duplicate texts, and creates group-aware 80/10/10 train/validation/test splits. It does **not** train a model.

**Expected output:** `train.csv.gz`, `validation.csv.gz`, `test.csv.gz`, `split_category_counts.csv`, and `split_manifest.json`.

The removed `marathi_hindi_mixed_script` category is intentionally excluded. If it appears in the data, the notebook stops for inspection rather than silently deleting rows.

In [ ]:
from google.colab import files
from pathlib import Path
import pandas as pd, numpy as np, zipfile, shutil, json, hashlib, re, unicodedata
from sklearn.model_selection import StratifiedGroupKFold

SEED = 42
ZIP_PATH = Path('/content/prepared_dataset.zip')
EXTRACT_DIR = Path('/content/prepared_dataset')
OUTPUT_DIR = Path('/content/processed_dataset')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not ZIP_PATH.exists():
    uploaded = files.upload()
    zips = [Path('/content') / name for name in uploaded if name.lower().endswith('.zip')]
    if not zips:
        raise FileNotFoundError('Upload prepared_dataset.zip')
    ZIP_PATH = zips[0]

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as zf:
    root = EXTRACT_DIR.resolve()
    for member in zf.infolist():
        target = (EXTRACT_DIR / member.filename).resolve()
        if target != root and root not in target.parents:
            raise ValueError(f'Unsafe ZIP path: {member.filename}')
    zf.extractall(EXTRACT_DIR)
print('Extracted:', EXTRACT_DIR)

## 1. Locate the combined dataset

The archive may contain multiple files. We list supported data files and detect tables with text plus category columns.

In [ ]:
EXTS = ('.csv', '.csv.gz', '.parquet', '.jsonl')
data_files = sorted(p for p in EXTRACT_DIR.rglob('*') if p.is_file() and p.name.lower().endswith(EXTS))
if not data_files:
    raise FileNotFoundError('No CSV/Parquet/JSONL files found in ZIP')

def read_table(path, nrows=None):
    name = path.name.lower()
    if name.endswith('.parquet'):
        frame = pd.read_parquet(path)
        return frame.head(nrows) if nrows else frame
    if name.endswith('.jsonl'):
        return pd.read_json(path, lines=True, nrows=nrows)
    return pd.read_csv(path, nrows=nrows, low_memory=False)

candidates = []
for i, path in enumerate(data_files):
    try:
        sample = read_table(path, nrows=5)
        cols = {str(c).strip().lower() for c in sample.columns}
        has_text = bool(cols & {'text','sentence','utterance','transcript'})
        has_category = bool(cols & {'category','category_name','language_category'})
        print(f'[{i}] {path.relative_to(EXTRACT_DIR)} | columns={list(sample.columns)}')
        if has_text and has_category:
            candidates.append(path)
    except Exception as exc:
        print('Skipped unreadable file:', path, repr(exc))
if not candidates:
    raise ValueError('No combined table with text and category columns detected.')
print('\nCandidate combined tables:')
for i, path in enumerate(candidates): print(i, path)
DATA_CHOICE = 0  # Change this if the intended combined table is not candidate 0.
DATA_PATH = candidates[DATA_CHOICE]
df = read_table(DATA_PATH)
print('Selected:', DATA_PATH, '| shape:', df.shape)
display(df.head(3))

## 2. Map categories to final language labels

Unknown categories and conflicting pre-existing labels stop the notebook for review. We do not silently assign labels.

In [ ]:
def find_col(frame, names, required=True):
    lookup = {str(c).strip().lower(): c for c in frame.columns}
    for name in names:
        if name in lookup: return lookup[name]
    if required: raise KeyError(f'Missing one of {names}; columns={list(frame.columns)}')
    return None

text_col = find_col(df, ['text','sentence','utterance','transcript'])
category_col = find_col(df, ['category','category_name','language_category'])
existing_label_col = find_col(df, ['label','language_label','target'], required=False)
df = df.rename(columns={text_col:'text', category_col:'category'}).copy()
df['text'] = df['text'].astype('string').fillna('').str.strip()
df['category'] = df['category'].astype('string').fillna('').str.strip().str.lower()
df = df[df['text'].ne('') & df['category'].ne('')].copy()

CATEGORY_TO_LABEL = {
 'assamese_only':'as-IN', 'assamese_english_mixed_script':'as-IN',
 'bengali_only':'bn-IN', 'bengali_english_mixed_script':'bn-IN',
 'english_only':'en-IN',
 'gujarati_only':'gu-IN', 'gujarati_english_mixed_script':'gu-IN',
 'hindi_only':'hi-IN', 'hindi_english_mixed_script':'hi-IN',
 'kannada_only':'kn-IN', 'kannada_english_mixed_script':'kn-IN',
 'malayalam_only':'ml-IN', 'malayalam_english_mixed_script':'ml-IN',
 'marathi_only':'mr-IN', 'marathi_english_mixed_script':'mr-IN',
 'punjabi_only':'pa-IN', 'punjabi_english_mixed_script':'pa-IN',
 'punjabi_hindi_mixed_script':'pa-IN', 'punjabi_hindi_english_mixed_script':'pa-IN',
 'tamil_only':'ta-IN', 'tamil_english_mixed_script':'ta-IN',
 'telugu_only':'te-IN', 'telugu_english_mixed_script':'te-IN',
 'unsupported_language_only':'none', 'unsupported_language_english_mixed_script':'none'
}
if df['category'].eq('marathi_hindi_mixed_script').any():
    raise ValueError('Found removed category marathi_hindi_mixed_script. Inspect the input ZIP; no rows were silently removed.')
unknown = sorted(set(df['category']) - set(CATEGORY_TO_LABEL))
if unknown: raise ValueError(f'Unmapped categories: {unknown}')
df['label'] = df['category'].map(CATEGORY_TO_LABEL)
if existing_label_col:
    old = df[existing_label_col].astype('string').fillna('').str.strip()
    bad = old.ne('') & old.ne(df['label'])
    if bad.any():
        display(df.loc[bad, ['text','category',existing_label_col,'label']].head(25))
        raise ValueError(f'{int(bad.sum())} pre-existing labels conflict with category-derived labels.')
print('Rows:', len(df), '| labels:', sorted(df['label'].unique()))
display(df.groupby(['label','category']).size().rename('rows').reset_index())

## 3. Deduplicate and construct split groups

Normalized exact duplicates are kept in one row only. Conflicting labels for the same normalized text stop the notebook. `source_group_id` is grouped within `source_file` when available; synthetic examples with no source group use `generator + template_id` as a conservative group. Generic template IDs may be coarse, so inspect the group count before continuing.

In [ ]:
def normalize_text(value):
    value = unicodedata.normalize('NFKC', str(value)).casefold()
    return re.sub(r'\s+', ' ', value).strip()
df['_normalized_text'] = df['text'].map(normalize_text)
df['_text_sha256'] = df['_normalized_text'].map(lambda s: hashlib.sha256(s.encode('utf-8')).hexdigest())
conflict = df.groupby('_text_sha256')['label'].nunique()
conflict_hashes = set(conflict[conflict > 1].index)
if conflict_hashes:
    display(df[df['_text_sha256'].isin(conflict_hashes)][['text','category','label']].head(50))
    raise ValueError(f'{len(conflict_hashes)} normalized texts have conflicting labels.')
rows_before_dedup = len(df)
df = df.drop_duplicates('_text_sha256', keep='first').copy()
print('Duplicate rows removed:', rows_before_dedup - len(df))

def val(row, col):
    if col not in row.index or pd.isna(row[col]): return ''
    s = str(row[col]).strip()
    return '' if s.lower() in {'','nan','none','null'} else s

def group_for(row):
    source = val(row, 'source_file') or val(row, 'source') or 'unknown_source'
    explicit = val(row, 'source_group_id')
    if explicit: return f'source::{source}::{explicit}'
    synthetic = val(row, 'is_synthetic').lower() in {'true','1','yes'}
    template, generator = val(row, 'template_id'), val(row, 'generator')
    if synthetic and template: return f'template::{generator}::{template}'
    return 'text::' + row['_text_sha256']

df['_split_group'] = df.apply(group_for, axis=1)
print('Unique groups:', df['_split_group'].nunique(), '| rows:', len(df))
print('Rows per label:')
display(df['label'].value_counts().sort_index().rename('rows').to_frame())
if df.groupby('_split_group')['label'].nunique().gt(1).any():
    raise ValueError('A split group contains multiple labels. Review source_group_id/template grouping before proceeding.')

## 4. Split 80/10/10 without group leakage

Ten stratified group folds are used: fold 0 is test, fold 1 is validation, and the remaining folds are training. Actual percentages can vary slightly because groups cannot be broken apart.

In [ ]:
EXPECTED = {'as-IN','bn-IN','en-IN','gu-IN','hi-IN','kn-IN','ml-IN','mr-IN','pa-IN','ta-IN','te-IN','none'}
if set(df['label'].unique()) != EXPECTED:
    raise ValueError(f'Expected 12 labels {sorted(EXPECTED)}, observed {sorted(df.label.unique())}')
if df['_split_group'].nunique() < 10:
    raise ValueError('Fewer than 10 independent groups; cannot perform 10-fold split.')
sgkf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=SEED)
fold = pd.Series(index=df.index, dtype='int64')
for fold_id, (_, held_idx) in enumerate(sgkf.split(df, df['label'], groups=df['_split_group'])):
    fold.iloc[held_idx] = fold_id
df['split'] = 'train'
df.loc[fold.eq(0), 'split'] = 'test'
df.loc[fold.eq(1), 'split'] = 'validation'

for name in ['train','validation','test']:
    labels_here = set(df.loc[df.split.eq(name), 'label'])
    missing = EXPECTED - labels_here
    if missing: raise ValueError(f'{name} missing labels: {sorted(missing)}. Inspect group structure.')
assert df.groupby('_split_group')['split'].nunique().max() == 1
assert df.groupby('_text_sha256')['split'].nunique().max() == 1
print('Leakage checks passed.')
display(pd.crosstab(df['split'], df['label'], margins=True))
display(df['split'].value_counts().rename('rows').to_frame().assign(percent=lambda x: (x.rows/len(df)*100).round(2)))

## 5. Export splits and audit files

Provenance and review fields are preserved when available. Review/synthetic flags are not treated as proof of data quality.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
keep_cols = ['text','label'] + [c for c in ['category','source','source_file','source_group_id','text_sha256','review_status','is_mixed_script_candidate','is_synthetic','generator','template_id','split'] if c in df.columns]
for name in ['train','validation','test']:
    path = OUTPUT_DIR / f'{name}.csv.gz'
    df.loc[df.split.eq(name), keep_cols].to_csv(path, index=False, compression='gzip')
    print(path.name, f'{path.stat().st_size/1024/1024:.2f} MiB')

counts = df.groupby(['split','label','category']).size().rename('rows').reset_index()
counts.to_csv(OUTPUT_DIR / 'split_category_counts.csv', index=False)
manifest = {
 'input_file': str(DATA_PATH), 'seed': SEED,
 'rows_after_deduplication': int(len(df)),
 'split_rows': {k:int(v) for k,v in df['split'].value_counts().items()},
 'label_names': sorted(df['label'].unique().tolist()),
 'duplicate_rows_removed': int(rows_before_dedup-len(df)),
 'group_leakage_check': 'passed', 'normalized_text_leakage_check': 'passed',
 'removed_category': 'marathi_hindi_mixed_script',
 'note': 'Review source licenses and review_status before model training.'
}
(OUTPUT_DIR / 'split_manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
with zipfile.ZipFile('/content/processed_dataset_bundle.zip','w',zipfile.ZIP_DEFLATED) as zf:
    for path in OUTPUT_DIR.iterdir():
        if path.is_file(): zf.write(path, arcname=path.name)
files.download('/content/processed_dataset_bundle.zip')